<a href="https://colab.research.google.com/github/monty313/the-truth/blob/main/GPU_EDITION/Momentum_One_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Momentum One — GPU Edition (Bot 1.5)
### Train thousands of markets at once — and never lose progress

**FIRST: turn on the GPU.**
Menu -> **Runtime** -> **Change runtime type** -> pick **L4 GPU** -> **Save**.

Then run the cells below, top to bottom.


## Cell 1 — load the bot + connect your Drive (run once)
This downloads the bot and connects your **Google Drive**, so the best brain is saved there and **survives Colab shutting off**.

A window will pop up asking permission to use your Google Drive — click **Allow / Connect**. That is how it remembers the best brain between sessions.


In [1]:
import torch, os, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 GPU, then re-run')
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/monty313/the-truth.git
%cd the-truth
!pip -q install pyyaml >/dev/null 2>&1
DR = '/content/drive/MyDrive/momentum_gpu'
os.makedirs(DR, exist_ok=True)
for f in os.listdir('artifacts/checkpoints'):
    if f.endswith('.pt') and not os.path.exists(f'{DR}/{f}'):
        shutil.copy(f'artifacts/checkpoints/{f}', f'{DR}/{f}')
shutil.rmtree('artifacts/checkpoints', ignore_errors=True)
os.symlink(DR, 'artifacts/checkpoints')
print('best brain saved to your Drive folder:', DR)
print('ready.')


GPU: NVIDIA L4
Mounted at /content/drive
/content/the-truth
best brain saved to your Drive folder: /content/drive/MyDrive/momentum_gpu
ready.


## Cell 2 — START TRAINING
Trains **8,000 markets at once**, each with a random target (2.5%-70.3%) and random risk (1%-4.4%).

It **runs until Colab stops it** (or until it wins at 365 in a row). The best brain is saved to your Drive every round, so nothing is ever lost.

**To continue another day:** just open this notebook again, run Cell 1, then this cell — it picks up from your best brain automatically.

If you ever see *out of memory*: change `--instances 8000` to `--instances 4000`.


In [ ]:
!python scripts/gpu_train.py --instances 8000 --minutes 1440 --patience 100000


## Cell 3 — how is it doing?
Best streak so far, and every saved record. `pass0012` = **12 cleared days in a row**.


In [ ]:
!cat artifacts/checkpoints/gpu_progress.json
print('\n--- saved records ---')
!ls -1 artifacts/checkpoints/ 2>/dev/null | grep gpu_pass || echo 'no records yet'


## Cell 4 — the good brain (already on your Drive)
Your best brain is already saved on your Google Drive at **MyDrive / momentum_gpu / gpu_best.pt** — nothing to download. This cell just copies it to your computer too, if you want a second copy.


In [ ]:
from google.colab import files
files.download('artifacts/checkpoints/gpu_best.pt')


## Cell 5 — double-check on the REAL sim (optional, ~5 min)
Runs the saved brain on your exact simulator (not the fast copy) so you can trust it.


In [ ]:
!python scripts/gpu_validate.py --csv data/XAUUSD_curriculum_2026.csv --ckpt gpu_best
